# SemanticSCVI (geometric) — atlas-wide CD4 (non-Treg) factors vs **malignancy**

Trains a geometric `SemanticSCVI` (Geneformer prior) on the curated combined skin-T atlas
(`skin_T_tcr_malig_v2.h5ad`, 242,959 cells) and inspects how its factors separate a chosen
malignancy call. Steps:

1. **Load** the atlas and **join all precomputed malignancy calls** — TCR ALICE (nb21),
   CNV all-T + CD4/CD8-ref (nb14/15), transcriptomic MrVI m1/m2 (nb16/17) — with a short summary.
2. **Pick the driving call** at the top via `MALIGNANCY_COL` (default `tcr_malignant_alice`);
   it becomes `is_malignant` and drives every panel below.
3. **Select samples** (de-dup / non-αβ-CD4 / size gate) → **subset to CD4 non-Treg**.
4. **Train / reload** the geometric `SemanticSCVI` (same knobs as nb12).
5. **Inspect separation**: latent UMAP, per-factor heatmaps, factor geometry + gene-direction planes.

Part 1 is CPU-light. Parts 2-3 (Geneformer map build, training, UMAP, DPT) are **HEAVY** — run on
the `neural_nmf_env` GPU kernel.

In [ ]:
# ============================================================
# Parameters — malignancy-call selector + SemanticSCVI knobs (geometric Geneformer, same as nb12).
# ============================================================
import hashlib
import json
from pathlib import Path


def _resolve_nb_dir() -> Path:
    start = Path.cwd()
    for base in [start, *start.parents]:
        for sub in [Path("."), Path("notebooks/MF"), Path("scvi-tools-neural-nmf/notebooks/MF")]:
            cand = base / sub
            if cand.name == "MF" and (cand / "data").exists():
                return cand.resolve()
    raise FileNotFoundError(f"could not locate MF/data starting from {start}")


def _semantic_cache_slug(kwargs, max_epochs, warmup_epochs, n_epochs_kl_warmup, hvg_top_n, n=10):
    """Stable 10-char hash of every param that affects the trained SemanticSCVI model."""
    blob = json.dumps(
        {
            "kwargs": dict(sorted(kwargs.items())),
            "max_epochs": max_epochs,
            "warmup_epochs": warmup_epochs,
            "n_epochs_kl_warmup": n_epochs_kl_warmup,
            "hvg_top_n": hvg_top_n,
        },
        default=str, sort_keys=True,
    )
    return hashlib.sha1(blob.encode()).hexdigest()[:n]


NB_DIR = _resolve_nb_dir()
print(f"NB_DIR = {NB_DIR}")

# ---- paths ----
V2             = NB_DIR / "data" / "atlas_joint" / "skin_T_tcr_malig_v2.h5ad"  # curated TCR-complete object
GENE_ID_SOURCE = NB_DIR / "data" / "cache" / "cnmf_malignant_counts.h5ad"   # gene_name -> gene_id
SEMANTIC_CACHE_GENEFORMER = NB_DIR / "data" / "mf_cd4_atlas_geneformer.pt"
MODEL_CACHE_DIR = NB_DIR / "models" / ".model_cache_semantic_geom_cd4_atlas"
FIG_DIR = NB_DIR / "figures"; FIG_DIR.mkdir(exist_ok=True)
EMB_UMAP_NPY = NB_DIR / "data" / "atlas_joint" / "semantic_geom_cd4_atlas_emb_umap.npy"

# ---- per-donor clone thresholds (drive the sample-selection rules in Part 1) ----
FRAC_THRESH, RATIO_THRESH, EXPANDED_MIN = 0.05, 1.33, 2

# ---- CD4 non-Treg selection: cell_type_T2 == "CD4" (excludes CD4_Treg / CD8 / NK) ----

# ---- malignancy call that drives every downstream panel (key in M.MALIGNANCY_SOURCES) ----
#   tcr_malignant_alice (default, nb21 TCR ALICE)
#   cnvT_cell_gmm / cnvT_cell_mrvi / cnvT_cnvcluster / cnvT_cnvcluster_mrvi  (nb14 CNV, all-T)
#   cnv_cd4_cd8ref            (nb15 CNV, CD4 vs same-sample CD8 ref — CD4-only coverage)
#   mrvi_m1_labelspread       (nb16 transcriptomic LabelSpreading)
#   mrvi_m2_pseudosample      (nb17 transcriptomic pseudosample)
MALIGNANCY_COL = "tcr_malignant_alice"

# ---- Preprocessing ----
HVG_TOP_N  = 2500
HVG_FLAVOR = "seurat_v3"

# ---- Shared model size ----
N_LATENT  = 10
BATCH_KEY = "sample_id"

# ---- SemanticSCVI (Geneformer prior) — same as nb12 ----
SEMANTIC_GENEFORMER_MAX_EPOCHS = 200
SEMANTIC_GENEFORMER_WARMUP_EPOCHS = 20
SEMANTIC_GENEFORMER_N_EPOCHS_KL_WARMUP = 100
SEMANTIC_GENEFORMER_KWARGS = dict(
    loss_mode="geometric",
    coherence_weight=1000.0,
    n_gene_sample=1024,
    n_latent=N_LATENT,
    n_layers=1,
    n_hidden=128,
    dropout_rate=0.1,
    gene_likelihood="nb",
    weights_positive=True,
    use_batch_norm=False,
)
SEMANTIC_GENEFORMER_CACHE_DIR = MODEL_CACHE_DIR / "semantic_scvi" / _semantic_cache_slug(
    {**SEMANTIC_GENEFORMER_KWARGS, "batch_key": BATCH_KEY},
    SEMANTIC_GENEFORMER_MAX_EPOCHS,
    SEMANTIC_GENEFORMER_WARMUP_EPOCHS,
    SEMANTIC_GENEFORMER_N_EPOCHS_KL_WARMUP,
    HVG_TOP_N,
)
print(f"semantic_geom cache_dir: {SEMANTIC_GENEFORMER_CACHE_DIR}")
FORCE_TRAIN_SEMANTIC_GENEFORMER = False

# ---- interpretation (ref values from nb12) ----
PER_PROJECTION_N_TOP = 30

In [ ]:
import sys, gc, importlib
if str(NB_DIR) not in sys.path:
    sys.path.insert(0, str(NB_DIR))            # MF/ holds atlas_join_helpers, skin_T_cnv_helpers, semantic_malig_helpers
if str(NB_DIR.parent) not in sys.path:
    sys.path.insert(0, str(NB_DIR.parent))     # notebooks/ holds benchmark_helpers

import numpy as np
import pandas as pd
import scanpy as sc
import torch
import scvi

import atlas_join_helpers as H
import skin_T_cnv_helpers as C
import semantic_malig_helpers as M
importlib.reload(H); importlib.reload(C); importlib.reload(M)
from benchmark_helpers import get_or_build_geneformer_map, train_or_load_semantic_scvi

assert MALIGNANCY_COL in M.MALIGNANCY_SOURCES, \
    f"MALIGNANCY_COL={MALIGNANCY_COL!r} not in {list(M.MALIGNANCY_SOURCES)}"

scvi.settings.seed = 0
np.random.seed(0)
sc.settings.verbosity = 1
sc.settings.figdir = FIG_DIR
print("scvi", scvi.__version__, "| scanpy", sc.__version__, "| CUDA:", torch.cuda.is_available())

## Part 1 — load curated atlas + all malignancy calls  · CPU-light

1. **Load** the curated TCR-complete combined skin-T object (`skin_T_tcr_malig_v2.h5ad`).
2. **Join all precomputed malignancy calls** (`M.load_malignancy_calls`): TCR ALICE (nb21),
   CNV all-T + CD4/CD8-ref (nb14/15), transcriptomic MrVI m1/m2 (nb16/17). Each lands as a
   bool obs column; print a short **summary** (prevalence + pairwise Jaccard).
3. **Select samples** (per-donor clone table): de-dup `D5__MFIVB`≡`D1__P303`, drop non-αβ-CD4
   entities (`MF_gamma_delta`, `CD8_aggressive_epidermotropic_CTCL`), size-gate
   `n_tcr_cells < 300` (keep all herrera2021).
4. **Subset** to CD4 non-Treg (`cell_type_T2 == "CD4"`), drop ribosomal genes, and set the
   driving label `is_malignant = adata.obs[MALIGNANCY_COL]` (HC forced benign).

In [ ]:
adata = sc.read_h5ad(V2)                       # curated TCR-complete T object (242,959 cells)

# Join every precomputed malignancy call (nb14-17 CNV/transcriptomic + nb21 TCR ALICE)
# onto obs by cell name; uncovered cells -> benign. Each becomes a bool obs column.
adata = M.load_malignancy_calls(adata, NB_DIR)
print("loaded:", adata.shape)

In [ ]:
# Summary of all malignancy calls: prevalence + pairwise Jaccard (whole skin-T set).
counts, jaccard = M.malignancy_summary(adata)
print("driving call: MALIGNANCY_COL =", MALIGNANCY_COL, "\n")
print("prevalence per call:")
print(counts)
print("\npairwise Jaccard:")
with pd.option_context("display.width", 200, "display.max_columns", None):
    print(jaccard)
counts

In [ ]:
# Sample-selection rules on the per-donor clone table (functional — sets the training cohort).
clone_summary = C.clone_summary_table(adata, FRAC_THRESH, RATIO_THRESH)
meta = (
    adata.obs[["donor", "study", "entity", "disease"]].astype(str)
    .groupby("donor", observed=True)
    .agg(lambda s: ", ".join(sorted(s[s != "nan"].unique())))
)
clone_summary = clone_summary.merge(meta, on="donor", how="left")

drop = {}
# 1. de-duplicate: D5__MFIVB == D1__P303 (identical clone profile) -> keep D5__MFIVB
if {"D5__MFIVB", "D1__P303"} <= set(clone_summary["donor"]):
    drop["D1__P303"] = "duplicate of D5__MFIVB"
# 2. drop non-alpha-beta-CD4 entities (TCRb caller is blind to these -> route to CNV)
DROP_ENTITIES = {"MF_gamma_delta", "CD8_aggressive_epidermotropic_CTCL"}
for d in clone_summary.loc[clone_summary["entity"].isin(DROP_ENTITIES), "donor"]:
    drop.setdefault(d, "non-ab-CD4 entity")
# 3. size gate: n_tcr_cells < 300, except study == herrera2021
small = (clone_summary["n_tcr_cells"] < 300) & (clone_summary["study"] != "herrera2021")
for d in clone_summary.loc[small, "donor"]:
    drop.setdefault(d, "n_tcr_cells < 300")

keep_donors = clone_summary.loc[~clone_summary["donor"].isin(drop), "donor"].tolist()
if drop:
    print("dropped:", drop)
adata = adata[adata.obs["donor"].isin(keep_donors)].copy()
print(f"samples kept: {len(keep_donors)} | cells: {adata.n_obs}")

In [ ]:
adata_cd4 = adata[adata.obs["cell_type_T2"].astype(str) == "CD4"].copy()   # CD4 non-Treg (excl. CD4_Treg/CD8/NK)

# drop ribosomal protein genes (RPS*/RPL*) — they dominate loadings and are uninformative here
ribo = adata_cd4.var_names.str.upper().str.startswith(("RPS", "RPL"))
print(f"removing {int(ribo.sum())} ribosomal protein genes (RPS*/RPL*)")
adata_cd4 = adata_cd4[:, ~ribo].copy()

adata_cd4.X = adata_cd4.layers["raw_counts"].copy()   # NB likelihood needs raw counts

# driving malignancy label (selected at top) — HC samples are benign by definition
is_mal = adata_cd4.obs[MALIGNANCY_COL].astype(bool).to_numpy()
hc = adata_cd4.obs["disease"].astype(str).eq("HC").to_numpy()
n_flip = int(is_mal[hc].sum())
is_mal[hc] = False
adata_cd4.obs["is_malignant"] = is_mal
print(f"CD4 non-Treg: {adata_cd4.shape}  | label = {MALIGNANCY_COL} (HC forced benign: {n_flip})")
print("is_malignant:", int(is_mal.sum()), "/", adata_cd4.n_obs)
print("by study:\n",
      adata_cd4.obs.groupby("study", observed=True)["is_malignant"].agg(["size", "sum"]))
del adata; gc.collect()

## Part 2 — SemanticSCVI (geometric, Geneformer prior)  · HEAVY (GPU kernel)

Map gene symbols → Ensembl (from `cnmf_malignant_counts.h5ad`), build/load the full-gene Geneformer
map, HVG-subset adata + map together, then train (or reload) the model. First run trains ~250k
cells on GPU; reruns hit the param-hashed cache.

In [ ]:
src = sc.read_h5ad(GENE_ID_SOURCE, backed="r")
sym2ens = dict(zip(src.var["gene_name"].astype(str), src.var["gene_id"].astype(str)))
adata_cd4.var["gene_id"] = [sym2ens.get(s, s) for s in adata_cd4.var_names.astype(str)]
n_mapped = int(sum(g.startswith("ENSG") for g in adata_cd4.var["gene_id"]))
print(f"gene_id mapped to Ensembl: {n_mapped}/{adata_cd4.n_vars}")
del src; gc.collect()

In [ ]:
# Build the FULL-gene Geneformer map first (cached). Then HVG-subset adata + map together.
# Delete the .pt to force a rebuild.
semantic_map = get_or_build_geneformer_map(
    adata_cd4, SEMANTIC_CACHE_GENEFORMER, var_id_key="gene_id",
)
print("semantic_map (raw):", tuple(semantic_map.shape))

# Stale-cache guard: rebuild if the cached map rows don't match the current full-gene adata.
if semantic_map.shape[0] != adata_cd4.n_vars:
    print(f"shape mismatch ({semantic_map.shape[0]} vs {adata_cd4.n_vars}) — rebuilding")
    SEMANTIC_CACHE_GENEFORMER.unlink()
    semantic_map = get_or_build_geneformer_map(
        adata_cd4, SEMANTIC_CACHE_GENEFORMER, var_id_key="gene_id",
    )
    print("semantic_map (rebuilt):", tuple(semantic_map.shape))

# Keep only genes with a real Geneformer prior (non-zero embedding row). Out-of-vocab
# genes (~all non-coding RP11-*/AC0*/LINC*/MIR*) are zero-filled by the map builder and
# carry no semantic signal — drop them BEFORE HVG so HVG ranks variance only among
# semantically-grounded (effectively protein-coding) genes.
in_vocab = (semantic_map.norm(dim=1) > 0).cpu().numpy()
print(f"in-vocab (non-zero Geneformer row): {int(in_vocab.sum())}/{adata_cd4.n_vars}")
adata_cd4 = adata_cd4[:, in_vocab].copy()
semantic_map = semantic_map[torch.as_tensor(in_vocab)]

if HVG_TOP_N is not None:
    sc.pp.highly_variable_genes(adata_cd4, n_top_genes=HVG_TOP_N, flavor=HVG_FLAVOR, subset=False)
    keep = adata_cd4.var["highly_variable"].values
    adata_cd4 = adata_cd4[:, keep].copy()
    semantic_map = semantic_map[torch.as_tensor(keep)]
print("after HVG:", adata_cd4.shape, "| semantic_map:", tuple(semantic_map.shape))

In [ ]:
model = train_or_load_semantic_scvi(
    adata_cd4,
    semantic_map,
    cache_dir=SEMANTIC_GENEFORMER_CACHE_DIR,
    force_train=FORCE_TRAIN_SEMANTIC_GENEFORMER,
    max_epochs=SEMANTIC_GENEFORMER_MAX_EPOCHS,
    warmup_epochs=SEMANTIC_GENEFORMER_WARMUP_EPOCHS,
    n_epochs_kl_warmup=SEMANTIC_GENEFORMER_N_EPOCHS_KL_WARMUP,
    batch_key=BATCH_KEY,
    **SEMANTIC_GENEFORMER_KWARGS,
)
print("trained:", model.is_trained)

## Part 3 — latent embedding, loadings, and malignancy separation

Mirrors nb12 Sections A.2–E with `is_malignant` (= the selected `MALIGNANCY_COL`) as the label.

In [ ]:
z = np.asarray(model.get_latent_representation())
ad_emb = sc.AnnData(z, obs=model.adata.obs.copy())

# Cache the latent UMAP (neighbors + umap are the slow part). Recompute on shape mismatch.
FORCE_UMAP = True
if (not FORCE_UMAP) and EMB_UMAP_NPY.exists() and \
        np.load(EMB_UMAP_NPY).shape[0] == ad_emb.n_obs:
    ad_emb.obsm["X_umap"] = np.load(EMB_UMAP_NPY)
    print("loaded cached emb UMAP", ad_emb.obsm["X_umap"].shape)
else:
    sc.pp.neighbors(ad_emb, use_rep="X", random_state=0)
    sc.tl.umap(ad_emb, random_state=0)
    np.save(EMB_UMAP_NPY, ad_emb.obsm["X_umap"])
    print("computed + cached emb UMAP ->", EMB_UMAP_NPY)

color = [c for c in ["study",  "cell_type_T", 'tcr_malignant_alice','cnvT_cnvcluster','mrvi_m2_pseudosample'] if c in ad_emb.obs]
sc.pl.umap(ad_emb, color=color, ncols=2, wspace=0.3, frameon=False)

In [ ]:
W = model.get_loadings().reindex(adata_cd4.var_names)   # genes x N_LATENT
n_factors = z.shape[1]
print("W (loadings):", W.shape, "| n_factors:", n_factors, "| columns:", list(W.columns))
W.head()

In [ ]:
# top loading genes per factor (proj_Z_k) — same as the ref notebook
top_per_factor = {
    f"proj_{col}": W[col].nlargest(PER_PROJECTION_N_TOP).index.tolist()
    for col in W.columns
}
top_df = pd.DataFrame(top_per_factor)
top_df.index = [f"top_{i + 1}" for i in range(PER_PROJECTION_N_TOP)]
with pd.option_context("display.max_columns", None, "display.width", 250,
                       "display.max_colwidth", 30):
    print(top_df)
top_df

### Malignancy calls on the latent

All calls were joined into `obs` in Part 1 and ride along on `ad_emb` (the latent AnnData).
The headline label is `is_malignant` (= the selected `MALIGNANCY_COL`, HC forced benign); the
other calls (`M.MALIGNANCY_SOURCES`) stay available as a method-comparison in the panels below.

In [ ]:
# Calls compared in the panels below: the driving label + every registry call present in obs.
CALLS = [c for c in M.MALIGNANCY_SOURCES if c in ad_emb.obs]
for c in CALLS:
    ad_emb.obs[c] = ad_emb.obs[c].astype(bool)
print("driving call:", MALIGNANCY_COL)
print("malignant counts:", {c: int(ad_emb.obs[c].sum()) for c in CALLS})

### Per-factor separation heatmaps (one per call)

For factor `Z_k`, cells are sorted by `Z_k` and binned along x; bin color = fraction of malignant
cells. A clean red↔blue gradient = a factor that separates that call. Class-balanced per call. Row
labels carry the per-factor AUROC.

In [ ]:
M.plot_factor_heatmaps(z, ad_emb, CALLS, FIG_DIR, prefix="semantic_geom_cd4_atlas")

### Factor geometry vs the driving call (`is_malignant` = `MALIGNANCY_COL`)

2-factor scatter grid, 3-D scatter, and a gene-direction plane sweep, all labeled by
`is_malignant`. **Re-pick `FACTORS_GRID` / `FX3,FY3,FZ3` from the per-factor `aucs` printed
below** — the defaults are nb12's picks and may not be the best separators for this call.

In [ ]:
MALIGNANCY_COL = "mrvi_m2_pseudosample"   # transcriptomic pseudosample (nb17)

CALL = MALIGNANCY_COL
POS_LABEL, NEG_LABEL = "malignant", "benign"
MAX_PER_CLASS = 20000   # class-balance cap per call

is_mal = ad_emb.obs["is_malignant"].astype(bool).to_numpy().astype(int)
y_true = is_mal
plot_idx = M.balanced_idx(is_mal, MAX_PER_CLASS)
aucs = M.call_aucs(z, is_mal)
print(f"{CALL}: {int(is_mal.sum())} malignant / {len(is_mal)} cells | "
      f"balanced subsample {len(plot_idx)} ({len(plot_idx) // 2}/class)")
print("per-factor AUROC:", {k: round(v, 3) for k, v in aucs.items()})

# top loading genes per factor (consumed by the gene-direction sweep below)
top_df = pd.DataFrame({
    f"proj_{col}": W[col].nlargest(PER_PROJECTION_N_TOP).index.tolist()
    for col in W.columns
})
top_df.index = [f"top_{i + 1}" for i in range(PER_PROJECTION_N_TOP)]

In [ ]:
FACTORS_GRID = [4, 5, 7, 9, 1]   # nb12 status picks; re-pick from `aucs` if other factors win
M.plot_factor_grid(z, is_mal, plot_idx, FACTORS_GRID, aucs, CALL, POS_LABEL, NEG_LABEL, FIG_DIR)

#### 3-D scatter — factors (Z_FX3, Z_FY3, Z_FZ3)

Diagonal score `s3 = z(Z_FX3) + z(Z_FY3) + z(Z_FZ3)`; left panel colored by the call, right by
`s3`, with median / Youden-best planes drawn at constant `s3`.

In [ ]:
FX3, FY3, FZ3 = 1, 9, 7   # nb12 picks; re-pick from `aucs`
geom = M.make_geom3d(z, FX3, FY3, FZ3, plot_idx)
M.plot_3d_threshold(geom, is_mal, CALL, POS_LABEL, NEG_LABEL, FIG_DIR)

In [ ]:
# 1) Pseudotime via DPT on the latent kNN graph. ad_emb's neighbors may have been skipped
#    when the UMAP was loaded from cache — rebuild the graph if missing.
if "neighbors" not in ad_emb.uns:
    sc.pp.neighbors(ad_emb, use_rep="X", random_state=0)
_root_mask = ~ad_emb.obs["is_malignant"].astype(bool).to_numpy()   # benign side as DPT root
assert _root_mask.any(), f"no benign cell for {CALL}"
root_idx = int(np.where(_root_mask)[0][0])
ad_emb.uns["iroot"] = root_idx
sc.tl.diffmap(ad_emb, random_state=0)
sc.tl.dpt(ad_emb)
pt = ad_emb.obs["dpt_pseudotime"].to_numpy()
pt = np.where(np.isfinite(pt), pt, np.nan)   # DPT can emit inf on disconnected components
print(f"pseudotime: root_idx={root_idx}  min={np.nanmin(pt):.3f}  "
      f"median={np.nanmedian(pt):.3f}  max={np.nanmax(pt):.3f}  n_nan={np.isnan(pt).sum()}")

In [ ]:
# 2) Editable gene list — default = top 4 from each of Z_1, Z_7, Z_6.
GENES_FOR_PLANE = (
    top_df["proj_Z_1"].head(4).tolist()
    + top_df["proj_Z_9"].head(4).tolist()
    + top_df["proj_Z_7"].head(4).tolist()
)
GENES_FOR_PLANE = list(dict.fromkeys(GENES_FOR_PLANE))  # de-dup, preserve order
missing = [g for g in GENES_FOR_PLANE if g not in adata_cd4.var_names]
assert not missing, f"missing in adata_cd4: {missing}"
print(f"{len(GENES_FOR_PLANE)} genes:", GENES_FOR_PLANE)

In [ ]:
# 3) Candidate direction vectors in (Z_FX3, Z_FY3, Z_FZ3) — OLS coeffs on standardized axes.
from sklearn.linear_model import LogisticRegression

Z3z = geom["Z3z"]   # standardized 3-axis subspace from make_geom3d
size_factor = np.asarray(adata_cd4.X.sum(axis=1)).ravel().astype(np.float64)
size_factor[size_factor == 0] = 1.0

dir_rows = [("diagonal_(1,1,1)", np.array([1.0, 1.0, 1.0]))]
dir_rows.append(("pseudotime", M.fit_direction(Z3z, pt)))
for g in GENES_FOR_PLANE:
    dir_rows.append((g, M.fit_direction(Z3z, M.gene_lognorm(adata_cd4, g, size_factor))))
lr = LogisticRegression(max_iter=1000, C=1.0).fit(Z3z, is_mal)
dir_rows.append((f"LR_optimal_(Z{FX3},Z{FY3},Z{FZ3})", lr.coef_.ravel().astype(np.float64)))

dirs = {name: v / max(np.linalg.norm(v), 1e-12) for name, v in dir_rows}
print(f"{len(dirs)} candidate directions:", list(dirs.keys()))

In [ ]:
# 4) Score each direction: AUROC + Youden-best threshold + accuracy stats.
results = (
    pd.DataFrame([M.eval_direction(n, v, Z3z, y_true) for n, v in dirs.items()])
    .sort_values("AUROC", ascending=False)
    .reset_index(drop=True)
)
with pd.option_context("display.max_columns", None, "display.width", 200,
                       "display.float_format", "{:.4f}".format):
    print(results.to_string(index=False))
out_csv = FIG_DIR / f"semantic_geom_cd4_atlas_{CALL}_gene_planes_stats.csv"
results.to_csv(out_csv, index=False)
print("saved", out_csv)
results

In [ ]:
# 5) 3-D scatter of the top-AUROC plane (left = label, right = marker-gene expression).
MARK_GENE = "KLF2" if "KLF2" in adata_cd4.var_names else GENES_FOR_PLANE[0]
mark_expr = M.gene_lognorm(adata_cd4, MARK_GENE, size_factor)
M.plot_best_plane(geom, results, is_mal, CALL, POS_LABEL, NEG_LABEL, mark_expr, MARK_GENE, FIG_DIR)

In [ ]:
# Interactive rotatable 3-D version of the top-AUROC plane (standalone .html).
M.plot_best_plane_plotly(geom, results, is_mal, CALL, POS_LABEL, NEG_LABEL, mark_expr, MARK_GENE, FIG_DIR)